# Final solution for Polimi RecSys challenge
Alessio Pizzini, David Ravelli

https://www.kaggle.com/competitions/recommender-systems-2025-challenge-polimi/leaderboard

## Retrieval (Candidate Generation) & Reranking with XGBoost

This notebook presents the final solution for the **RecSys Challenge 2025/26** developed by **Team Polenta** (1st place in both Private and Public leaderboards). The system implements a two-stage architecture designed to handle a dataset of over 3 million implicit interactions.

### Key Points:
* **Architecture**: A two-stage pipeline involving **Multi-Model Candidate Generation** followed by an **XGBoost Reranker**.
* **Recall Optimization**: Implemented a **Stepwise Greedy Selection** strategy to optimize the "Recall Ceiling," ensuring the target item is present in the candidate set.
* **Engineering for Scale**: Leveraged the **Numba** library for Just-In-Time (JIT) compilation to parallelize the extraction of over 90 complex features across millions of user-item pairs.
* **Robustness**: Utilized a **5-Fold K-Fold Bagging** strategy for XGBoost, achieving an outstanding model stability with a Pearson correlation of **0.998** between folds.
* **Performance**: Final Public Recall@20: **0.53331**; Private Recall@20: **0.53149**.

In [ ]:
!pip install protobuf==3.20.3
!pip install lightfm
!pip install implicit

# Preparation and Base Models

## Data Loading

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

INPUT_DIR = "/kaggle/input/recommender-systems-2025-challenge-polimi"

print("File disponibili in input:")
for dirname, _, filenames in os.walk(INPUT_DIR):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(f"{INPUT_DIR}/data_train.csv")

print(df.shape)
print(df.info())
df.head()

In [ ]:
!git clone https://github.com/remaplab/RecSys_Course_AT_PoliMi

In [ ]:
import sys
sys.path.append('./RecSys_Course_AT_PoliMi')
print(sys.path)

## Data Splitting Strategy
To train a reranker without data leakage, we adopted a multi-level split:
1.  **Model Training Set (80%)**: Used to train the base recommenders (SLIM, EASE_R, iALS, etc.).
2.  **Reranker Training Set (20%)**: Used as the ground truth for XGBoost to learn how to rank candidates generated by the base models.
3.  **K-Fold Cross-Validation**: The XGBoost set was further divided into 5 folds to ensure consistency and prevent overfitting to the private leaderboard.

In [ ]:
from scipy.sparse import coo_matrix, csr_matrix

n_users = df['row'].max() + 1
n_items = df['col'].max() + 1

print("Numero di utenti unici:", len(df['row'].unique()))

A_coo = coo_matrix((np.ones(len(df), dtype=np.int8), (df['row'], df['col'])),
                   shape=(n_users, n_items))

URM_all = A_coo.tocsr()

In [ ]:
from Evaluation.Evaluator import EvaluatorHoldout
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample

print(f"URM_all total interactions: {URM_all.nnz}")

# 80% base_models_train, 20% xgb_train
URM_train, URM_xgb = split_train_in_two_percentage_global_sample(URM_all, train_percentage=0.8)

print(f"URM_train: {URM_train.nnz} interactions")
print(f"URM_xgb:   {URM_xgb.nnz} interactions")

## Base Models Training on 80%

In [ ]:
#LIGHT FM
from Recommenders.FactorizationMachines.LightFMRecommender import LightFMCFRecommender

lightFM_params = {
                 'loss': 'warp',
                 'n_components': 157,
                 'item_alpha': 0.001019490618869453,
                 'user_alpha': 2.2744836269942957e-05,
                 'learning_rate': 0.012460981535727717,
                 'sgd_mode': 'adadelta',
                 'epochs': 74
              }

lightFM_recommender = LightFMCFRecommender(URM_train)
lightFM_recommender.fit(**lightFM_params)

In [ ]:
# MultVAE

import time
import torch
import torch.nn.functional as F
import numpy as np
import math
from tqdm import tqdm

from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch

try:
    from Recommenders.Neural.architecture_utils import generate_autoencoder_architecture
except ImportError:
    def generate_autoencoder_architecture(encoding_size, n_items, next_layer_size_multiplier, max_parameters, max_n_hidden_layers):
        enc_dims = [n_items]
        for _ in range(max_n_hidden_layers):
            next_dim = int(enc_dims[-1] / next_layer_size_multiplier)
            if next_dim < encoding_size: break
            enc_dims.append(next_dim)
        enc_dims.append(int(encoding_size))
        return enc_dims

# --- WRAPPER CLASS ---
class MultVAERecommender_PyTorch_Fixed(MultVAERecommender_PyTorch):

    def fit(self, epochs=10, batch_size=500, total_anneal_steps=200000,
            learning_rate=1e-3, l2_reg=0.01, dropout=0.5, anneal_cap=0.2,
            sgd_mode="adam", encoding_size=50, next_layer_size_multiplier=2,
            max_parameters=np.inf, max_n_hidden_layers=3, **earlystopping_kwargs):

        encoding_size = int(encoding_size)
        batch_size = int(batch_size)
        total_anneal_steps = int(total_anneal_steps)

        p_dims = generate_autoencoder_architecture(encoding_size, self.n_items, next_layer_size_multiplier, max_parameters, max_n_hidden_layers)
        p_dims = [int(d) for d in p_dims] # Fix lista int

        self._print(f"Architecture: {p_dims}")

        super().fit(epochs=epochs, batch_size=batch_size, dropout=dropout,
                    learning_rate=learning_rate, total_anneal_steps=total_anneal_steps,
                    anneal_cap=anneal_cap, p_dims=p_dims, l2_reg=l2_reg,
                    sgd_mode=sgd_mode, **earlystopping_kwargs)

    def _run_epoch(self, num_epoch):
        num_batches_per_epoch = math.ceil(len(self.warm_user_ids) / self.batch_size)
        if self.verbose:
            batch_iterator = tqdm(range(0, num_batches_per_epoch))
        else:
            batch_iterator = range(0, num_batches_per_epoch)

        self._model.train()
        epoch_loss = 0

        for _ in batch_iterator:
            self._optimizer.zero_grad()

            u_indices_np = np.random.choice(self.warm_user_ids, size=self.batch_size)
            user_batch_matrix = self.URM_train[u_indices_np]

            user_batch_tensor = torch.sparse_csr_tensor(
                user_batch_matrix.indptr, user_batch_matrix.indices, user_batch_matrix.data,
                size=user_batch_matrix.shape, dtype=torch.float32, device=self.device, requires_grad=False
            ).to_dense()
            # -----------------------------------------------------------

            logits, KL, mu_q, std_q, epsilon, sampled_z = self._model.forward(user_batch_tensor)
            log_softmax_var = F.log_softmax(logits, dim=1)
            neg_ll = - torch.mean(torch.sum(log_softmax_var * user_batch_tensor, dim=1))
            l2_reg_loss = self._model.get_l2_reg()

            if self.total_anneal_steps > 0:
                anneal = min(self.anneal_cap, 1. * self.update_count / self.total_anneal_steps)
            else:
                anneal = self.anneal_cap

            loss = neg_ll + anneal * KL + l2_reg_loss * self.l2_reg
            self.update_count += 1
            loss.backward()
            epoch_loss += loss.item()
            self._optimizer.step()

        if self.verbose:
            self._print("Loss {:.2E}".format(epoch_loss))
        self._model.eval()

    def _compute_item_score(self, user_id_array, items_to_compute=None):

        user_batch_matrix = self.URM_train[user_id_array]

        user_batch_tensor = torch.sparse_csr_tensor(
            user_batch_matrix.indptr, user_batch_matrix.indices, user_batch_matrix.data,
            size=user_batch_matrix.shape, dtype=torch.float32, device=self.device, requires_grad=False
        ).to_dense()
        # -----------------------------------------------------------------------------------

        with torch.no_grad():
            self._model.eval()
            logits, _, _, _, _, _ = self._model.forward(user_batch_tensor)

        item_scores_to_compute = logits.cpu().detach().numpy()

        if items_to_compute is not None:
            item_scores = - np.ones((len(user_id_array), self.n_items)) * np.inf
            item_scores[:, items_to_compute] = item_scores_to_compute[:, items_to_compute]
        else:
            item_scores = item_scores_to_compute

        return item_scores

In [ ]:
# Best parameters
vae_params = {
    'learning_rate': 0.0003696840239724747,
     'l2_reg': 0.0006675069550995708,
     'batch_size': 256,
     'dropout': 0.36764426901664576,
     'total_anneal_steps': 18522,
     'anneal_cap': 0.14211955419238365,
     'encoding_size': 545,
     'next_layer_size_multiplier': 10,
     'max_n_hidden_layers': 1
    }

multVAE_recommender = MultVAERecommender_PyTorch_Fixed(URM_train, use_gpu=True)

multVAE_recommender.fit(
    epochs=65,
    learning_rate=vae_params['learning_rate'],
    l2_reg=vae_params['l2_reg'],
    batch_size=int(vae_params['batch_size']),
    dropout=vae_params['dropout'],
    anneal_cap=vae_params['anneal_cap'],
    total_anneal_steps=int(vae_params['total_anneal_steps']),
    encoding_size=int(vae_params['encoding_size']),
    next_layer_size_multiplier=int(vae_params['next_layer_size_multiplier']),
    max_n_hidden_layers=int(vae_params['max_n_hidden_layers']),
    sgd_mode='adam'
)

print("Training MultVAE completato con i Best Params.")

In [ ]:
# RP3beta - GRAPH SHARPNER

from Recommenders.Recommender_utils import check_matrix, similarityMatrixTopK
from Recommenders.Similarity.Compute_Similarity_Python import Incremental_Similarity_Builder
from Utils.seconds_to_biggest_unit import seconds_to_biggest_unit
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender

rp3bSharp_params = {"alpha": 1.3, "beta": 0.6, "topK": 200, "normalize_similarity": True}

RP3BetaSharp_recommender = RP3betaRecommender(URM_train)
RP3BetaSharp_recommender.fit(
    **rp3bSharp_params,
    min_rating = 0,
    implicit = False,
)

In [ ]:
# RP3beta - HIGH PRECISION

from Recommenders.Recommender_utils import check_matrix, similarityMatrixTopK
from Recommenders.Similarity.Compute_Similarity_Python import Incremental_Similarity_Builder
from Utils.seconds_to_biggest_unit import seconds_to_biggest_unit
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender

rp3bHighPrec_params = {"alpha": 0.3, "beta": 0.15, "topK": 40, "normalize_similarity": True}

RP3BetaHighPrec_recommender = RP3betaRecommender(URM_train)
RP3BetaHighPrec_recommender.fit(
    **rp3bHighPrec_params,
    min_rating = 0,
    implicit = False,
)

In [ ]:
# iALS (implicit)

#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import numpy as np
import scipy.sparse as sps

try:
    from Recommenders.BaseRecommender import BaseRecommender
except ImportError:
    from BaseRecommender import BaseRecommender



class ImplicitALSRecommender(BaseRecommender):
    RECOMMENDER_NAME = "ImplicitALSRecommender"

    def __init__(self, URM_train, verbose=True):
        super(ImplicitALSRecommender, self).__init__(URM_train, verbose=verbose)

    def fit(self,
            factors=64,
            regularization=0.01,
            iterations=20,
            use_gpu=False,
            dtype=np.float32,
            calculate_training_loss=False,
            random_state=42,
            alpha_val=1.0):

        try:
            from implicit.als import AlternatingLeastSquares
        except ImportError as e:
            raise ImportError("Install the 'implicit' package (pip install implicit). Error: {}".format(e))


        self.factors = factors
        self.regularization = regularization
        self.iterations = iterations
        self.use_gpu = use_gpu
        self.dtype = dtype
        self.calculate_training_loss = calculate_training_loss
        self.random_state = random_state
        self.alpha_val = alpha_val

        # -----------------------------------------------------------------------
        # 1. DATA PREPARATION
        # -----------------------------------------------------------------------
        # implicit vuole (Item x User) per l'algoritmo ALS standard.
        # URM_train è (User x Item), quindi facciamo la trasposta.

        URM_train_csr = self.URM_train.copy().astype(self.dtype)
        item_user_data = URM_train_csr.T.tocsr()

        if self.alpha_val is not None and self.alpha_val != 1.0:
            item_user_data.data *= self.alpha_val

        # -----------------------------------------------------------------------
        # 2. MODEL SETUP & TRAINING
        # -----------------------------------------------------------------------
        self.model = AlternatingLeastSquares(
            factors=self.factors,
            regularization=self.regularization,
            iterations=self.iterations,
            calculate_training_loss=self.calculate_training_loss,
            use_gpu=self.use_gpu,
            random_state=self.random_state
        )

        if self.verbose:
            print(f"{self.RECOMMENDER_NAME}: Training with factors={self.factors}, alpha={self.alpha_val}...")

        self.model.fit(item_user_data)


        factor_A = self.model.user_factors
        factor_B = self.model.item_factors

        if self.use_gpu:
            try:
                import cupy as cp
                if isinstance(factor_A, cp.ndarray): factor_A = cp.asnumpy(factor_A)
                if isinstance(factor_B, cp.ndarray): factor_B = cp.asnumpy(factor_B)
            except ImportError:
                pass

        factor_A = factor_A.astype(np.float32)
        factor_B = factor_B.astype(np.float32)


        # Shape check
        if factor_A.shape[0] == self.n_users:
            self.user_factors = factor_A
            self.item_factors = factor_B
        elif factor_A.shape[0] == self.n_items:
            self.user_factors = factor_B
            self.item_factors = factor_A
        else:
            self.user_factors = factor_A
            self.item_factors = factor_B

        if self.verbose:
            print(f"{self.RECOMMENDER_NAME}: Training complete.")


    def _compute_item_score(self, user_id_array, items_to_compute=None):
        batch_user_factors = self.user_factors[user_id_array]  # Shape: (batch_size, n_factors)

        if items_to_compute is not None:
            batch_item_factors = self.item_factors[items_to_compute]
        else:
            batch_item_factors = self.item_factors

        scores = np.dot(batch_user_factors, batch_item_factors.T)

        return scores


    def save_model(self, folder_path, file_name=None):
        """
        Salvataggio compatibile con il framework DataIO
        """
        if file_name is None:
            file_name = self.RECOMMENDER_NAME

        self._print("Saving model to file '{}'".format(folder_path + file_name))

        data_dict = {
            "user_factors": self.user_factors,
            "item_factors": self.item_factors,
            "factors": self.factors,
            "regularization": self.regularization,
            "alpha_val": self.alpha_val
        }

        try:
            from Recommenders.DataIO import DataIO
            dataIO = DataIO(folder_path=folder_path)
            dataIO.save_data(file_name=file_name, data_dict_to_save=data_dict)
        except ImportError:
            np.savez(folder_path + file_name, **data_dict)

        self._print("Saving complete")

    def load_model(self, folder_path, file_name=None):
        if file_name is None:
            file_name = self.RECOMMENDER_NAME

        self._print("Loading model from file '{}'".format(folder_path + file_name))

        data_dict = None

        try:
            from Recommenders.DataIO import DataIO
            dataIO = DataIO(folder_path=folder_path)
            data_dict = dataIO.load_data(file_name=file_name)
        except (ImportError, FileNotFoundError):
            print("DataIO not found or file not found via DataIO, trying numpy.load")
            try:
                data_dict = np.load(folder_path + file_name + ".npz")
            except:
                data_dict = np.load(folder_path + file_name)

        self.user_factors = data_dict["user_factors"]
        self.item_factors = data_dict["item_factors"]
        self.factors = int(data_dict["factors"])

        if "regularization" in data_dict:
            self.regularization = data_dict["regularization"]
        if "alpha_val" in data_dict:
            self.alpha_val = data_dict["alpha_val"]

        self._print("Loading complete")

In [ ]:
iALS_recommender = ImplicitALSRecommender(URM_train, verbose=True)
iALS_recommender.fit(
    factors=63,
    regularization=7.535615818925292e-05,
    iterations=82,
    use_gpu=False,
    alpha_val=9.605102175195002,
    dtype=np.float32
)

In [ ]:
#SLIM EN

from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender

SLIM_EN_recommender = SLIMElasticNetRecommender(URM_train)
SLIM_EN_recommender.fit(
    l1_ratio = 0.21208828301360144,
    alpha = 0.0010731928833306275,
    positive_only = True,
    topK = 494
)

In [ ]:
#SLIM EN - NEGATIVES

from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender

slimENneg_params = {
            "topK": 200,
            "l1_ratio": 0.1,
            "alpha": 0.001,
            "positive_only": False
        }

SLIM_EN_neg_recommender = SLIMElasticNetRecommender(URM_train)
SLIM_EN_neg_recommender.fit(
    **slimENneg_params
)



In [ ]:
#EASE R

from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender

EASE_R_recommender = EASE_R_Recommender(URM_train)
EASE_R_recommender.fit(topK=725, l2_norm=375.6601396119732, normalize_matrix=False)


In [ ]:
#ItemKNN + TFIDF (implicit)

#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import numpy as np
import scipy.sparse as sps

try:
    from Recommenders.BaseRecommender import BaseRecommender
except ImportError:
    from BaseRecommender import BaseRecommender

try:
    from implicit.nearest_neighbours import TFIDFRecommender
except ImportError as e:
    raise ImportError("Install the 'implicit' package (pip install implicit). Error: {}".format(e))


class ImplicitTFIDFRecommender(BaseRecommender):
    """
    Wrapper for Implicit TFIDFRecommender compatible with Dacrema's Framework.
    """
    RECOMMENDER_NAME = "ImplicitTFIDFRecommender"

    def __init__(self, URM_train, verbose=True):
        super(ImplicitTFIDFRecommender, self).__init__(URM_train, verbose=verbose)

    def fit(self, K=20, num_threads=0):
        self.K = K
        self.num_threads = num_threads

        self.model = TFIDFRecommender(K=self.K, num_threads=self.num_threads)

        if self.URM_train.shape[0] == self.n_users:
            train_data = self.URM_train.tocsr()
        else:
            train_data = self.URM_train.T.tocsr()

        self.model.fit(train_data)

        self.W_sparse = self.model.similarity

        if self.W_sparse.shape[0] != self.n_items:
             raise ValueError(f"Error: W_sparse size ({self.W_sparse.shape}) doesn't match n_items ({self.n_items}). "
                              "Check URM orientation.")

        if self.verbose:
            print(f"{self.RECOMMENDER_NAME}: Training complete. W_sparse shape: {self.W_sparse.shape}")

    def _compute_item_score(self, user_id_array, items_to_compute=None):
        user_profile_batch = self.URM_train[user_id_array]

        if items_to_compute is not None:
            scores = user_profile_batch.dot(self.W_sparse[:, items_to_compute]).toarray()
        else:
            scores = user_profile_batch.dot(self.W_sparse).toarray()

        return scores




TFIDF_recommender = ImplicitTFIDFRecommender(URM_train, verbose=True)
TFIDF_recommender.fit(K=7, num_threads=0)

In [ ]:
#PureSVD

from sklearn.feature_extraction.text import TfidfTransformer
from Recommenders.MatrixFactorization.PureSVDRecommender import PureSVDRecommender
import numpy as np
import scipy.sparse as sps

def start_bm25_transformation(X, k1=1.2, b=0.75):
    X = X.copy()

    N = X.shape[0]
    item_freqs = np.diff(X.tocsc().indptr)
    idf = np.log(N / (item_freqs + 1.0) + 1.0)

    doc_len = np.diff(X.indptr)
    avg_dl = doc_len.mean()

    per_doc_denom = k1 * (1 - b + b * (doc_len / avg_dl))

    per_doc_denom_expanded = np.repeat(per_doc_denom, np.diff(X.indptr))

    tf = X.data

    idf_expanded = idf[X.indices]

    new_data = idf_expanded * ((tf * (k1 + 1)) / (tf + per_doc_denom_expanded))

    X.data = new_data
    return X


k1 = 1.996904472023335
b = 0.12355408697503084
URM_train_transformed = start_bm25_transformation(URM_train, k1=k1, b=b)

PureSVD_recommender = PureSVDRecommender(URM_train_transformed, verbose=True)

PureSVD_recommender.fit(
    num_factors=121,
    random_seed=1234
)

# XGB URM_xgb Dataset Preparation

## Stage 1: Candidate Generation (Retrieval)
The goal of this stage is to maximize the **Recall Ceiling** while maintaining a manageable number of candidates for the second stage. We selected a diverse ensemble of models including:
* **Linear Models**: SLIM Elastic Net (standard and negative) and EASE_R.
* **Graph-based**: RP3beta (Sharp and High Precision versions).
* **Matrix Factorization**: iALS and LightFM.
* **Neural Approaches**: Mult-VAE.

Each model was assigned a **personalized cutoff** obtained with Greedy Selection (e.g., 140 for SLIM), based on its marginal precision performance.

In [ ]:
import pandas as pd
import numpy as np
import scipy.sparse as sps
from tqdm import tqdm

# ==============================================================================
# 1. CANDIDATES GENERATION FUNCTION
# ==============================================================================
def get_candidates_with_labels(target_users, URM_target_labels, candidates_generators_dict, cutoff=None):
    """
    Genera il dataset per XGBoost usando i 'candidati' proposti dai modelli base.
    Gestisce cutoff specifici per modello se passati come tupla (Model, Cutoff).
    """
    URM_target_csr = URM_target_labels.tocsr()

    users_list = []
    items_list = []
    labels_list = []

    print(f"Generazione candidati per {len(target_users)} utenti...")

    for user_id in tqdm(target_users, desc="Candidate Gen"):
        candidates_set = set()

        for name, entry in candidates_generators_dict.items():
            if isinstance(entry, tuple):
                model, specific_cutoff = entry
            else:
                model = entry
                specific_cutoff = cutoff

            try:
                res = model.recommend(user_id, cutoff=specific_cutoff)
                if isinstance(res, tuple): res = res[0]
                candidates_set.update(res)
            except Exception as e:
                continue

        start_pos = URM_target_csr.indptr[user_id]
        end_pos = URM_target_csr.indptr[user_id+1]
        true_items = URM_target_csr.indices[start_pos:end_pos]

        candidates_list = list(candidates_set)

        if len(candidates_list) > 0:
            labels = np.isin(candidates_list, true_items).astype(int)
            users_list.extend([user_id] * len(candidates_list))
            items_list.extend(candidates_list)
            labels_list.extend(labels)

    return pd.DataFrame({
        "user_id": users_list,
        "item_id": items_list,
        "label": labels_list
    })

## High-Performance Feature Engineering with Numba
Calculating statistical similarity features (Skewness, Kurtosis, Mean, Std Dev) between candidate items and the user's history requires iterating over millions of rows.

To overcome Python's performance bottlenecks and the Global Interpreter Lock (GIL), we implemented the feature extraction logic using **Numba's `@njit(parallel=True)`**. This allows for:
* **Parallel Execution**: Utilizing all available CPU cores for cycle computation.
* **JIT Compilation**: Converting Python code into optimized machine code for high-speed execution.
* **Advanced Signals**: Capturing "Strong Match" signals via High Skewness or Max Similarity values.

In [ ]:
import pandas as pd
import numpy as np
import scipy.sparse as sps
from scipy.stats import skew, kurtosis
from tqdm import tqdm
from numba import njit, prange

# -----------------------------------------------------------------------
# HELPER NUMBA
# -----------------------------------------------------------------------
@njit
def _binary_search(arr, val, start, end):
    """Helper per trovare se un valore esiste in un array ordinato (User History)"""
    l = start
    r = end - 1
    while l <= r:
        mid = (l + r) // 2
        mid_val = arr[mid]
        if mid_val == val:
            return True
        elif mid_val < val:
            l = mid + 1
        else:
            r = mid - 1
    return False

@njit(parallel=True)
def compute_sim_stats_numba(users, items,
                            urm_indices, urm_indptr,
                            W_indices, W_indptr, W_data,
                            compute_skew_kurt=True):
    """
    Calcola feature statistiche.
    Se compute_skew_kurt=False, skew e kurtosis rimangono a 0.
    """
    n_samples = len(users)
    feat_max = np.zeros(n_samples, dtype=np.float32)
    feat_min = np.zeros(n_samples, dtype=np.float32)
    feat_mean = np.zeros(n_samples, dtype=np.float32)
    feat_std = np.zeros(n_samples, dtype=np.float32)
    feat_skew = np.zeros(n_samples, dtype=np.float32)
    feat_kurt = np.zeros(n_samples, dtype=np.float32)

    # Loop parallelo
    for k in prange(n_samples):
        u = users[k]
        candidate_item = items[k]

        start_u = urm_indptr[u]
        end_u = urm_indptr[u+1]

        if end_u - start_u == 0: continue

        start_i = W_indptr[candidate_item]
        end_i = W_indptr[candidate_item+1]

        if end_i - start_i == 0: continue

        count = 0
        sum_val = 0.0
        sum_sq_val = 0.0

        sum_cube_val = 0.0
        sum_four_val = 0.0

        max_val = -1.0e10
        min_val = 1.0e10

        for idx in range(start_i, end_i):
            neighbor = W_indices[idx]
            weight = W_data[idx]

            if _binary_search(urm_indices, neighbor, start_u, end_u):
                count += 1

                sum_val += weight
                sum_sq_val += (weight * weight)

                if weight > max_val: max_val = weight
                if weight < min_val: min_val = weight

                if compute_skew_kurt:
                    w2 = weight * weight
                    sum_cube_val += (w2 * weight)
                    sum_four_val += (w2 * w2)

        if count > 0:
            feat_max[k] = max_val
            feat_min[k] = min_val

            mu = sum_val / count
            feat_mean[k] = mu

            var = (sum_sq_val / count) - (mu * mu)

            if var > 1e-9:
                sigma = np.sqrt(var)
                feat_std[k] = sigma

                if compute_skew_kurt:
                    mean_cube = sum_cube_val / count
                    moment3 = mean_cube - 3 * mu * var - (mu * mu * mu)
                    feat_skew[k] = moment3 / (sigma * sigma * sigma)

                    mean_four = sum_four_val / count
                    mean_sq = sum_sq_val / count

                    moment4 = mean_four - 4*mu*mean_cube + 6*mu*mu*mean_sq - 3*mu*mu*mu*mu
                    feat_kurt[k] = (moment4 / (var * var)) - 3.0
            else:
                feat_std[k] = 0.0

    return feat_max, feat_min, feat_mean, feat_std, feat_skew, feat_kurt

# -----------------------------------------------------------------------
# FEATURES GENERATION FUNCTION
# -----------------------------------------------------------------------

def add_features_advanced(df_input, recommenders_dict, URM_train_profile, svd_model=None, n_latent_factors=5):
    """
    Calcola feature avanzate per XGBoost.
    OTTIMIZZATA UNIVERSALE CON NUMBA.
    """
    df = df_input.copy()
    n_rows = len(df)
    user_ids = df['user_id'].values
    item_ids = df['item_id'].values

    if not isinstance(URM_train_profile, sps.csr_matrix):
        URM_csr = URM_train_profile.tocsr()
    else:
        URM_csr = URM_train_profile

    URM_csr.sort_indices()


    all_norm_scores = []
    all_ranks = []

    # Batch sizes
    BATCH_SIZE_DEFAULT = 100000
    BATCH_SIZE_SLIM = 5000

    for name, model in recommenders_dict.items():
        print(f"   -> Elaborazione Modello: {name}", flush=True)

        # ---------------------------------------------------------------------
        # 1. RILEVAMENTO TIPO MODELLO
        # ---------------------------------------------------------------------
        is_lightfm = hasattr(model, 'RECOMMENDER_NAME') and 'LightFM' in model.RECOMMENDER_NAME
        has_factors = hasattr(model, 'user_factors') and hasattr(model, 'item_factors')

        W_matrix = None
        is_dense_matrix = False

        possible_attrs = ['W_sparse', 'W', 'B']
        for attr in possible_attrs:
            if hasattr(model, attr):
                temp_W = getattr(model, attr)
                if temp_W is not None:
                    W_matrix = temp_W
                    if isinstance(W_matrix, np.ndarray):
                        is_dense_matrix = True
                    break

        is_item_item = (W_matrix is not None)

        # ---------------------------------------------------------------------
        # 2. CALCOLO SCORE E RANK (STANDARD)
        # ---------------------------------------------------------------------
        current_batch_size = BATCH_SIZE_SLIM if is_item_item else BATCH_SIZE_DEFAULT

        if f'score_{name}' not in df.columns:
            scores_col = np.zeros(n_rows, dtype=np.float32)

            if is_item_item and not is_dense_matrix:
                try:
                    if not isinstance(W_matrix, sps.csc_matrix):
                        W_matrix = W_matrix.tocsc()
                except:
                    W_matrix = None

            # --- LOOP BATCH ---
            iterator = range(0, n_rows, current_batch_size)
            with tqdm(total=len(iterator), desc=f"      Score calc ({name})", leave=False) as pbar:
                for start in iterator:
                    end = min(start + current_batch_size, n_rows)
                    u_batch = user_ids[start:end]
                    i_batch = item_ids[start:end]

                    if is_lightfm:
                        try: scores_col[start:end] = model.predict(u_batch, i_batch)
                        except: scores_col[start:end] = model.lightFM_model.predict(u_batch, i_batch)
                    elif has_factors:
                        u_factors = model.user_factors[u_batch]
                        i_factors = model.item_factors[i_batch]
                        scores_col[start:end] = np.sum(u_factors * i_factors, axis=1)
                    elif W_matrix is not None:
                        try:
                            unique_items, inverse_indices = np.unique(i_batch, return_inverse=True)
                            W_subset = W_matrix[:, unique_items]
                            user_profile_batch = model.URM_train[u_batch]
                            batch_scores_subset = user_profile_batch.dot(W_subset)

                            if sps.issparse(batch_scores_subset): batch_scores_dense = batch_scores_subset.toarray()
                            else: batch_scores_dense = batch_scores_subset

                            scores_col[start:end] = batch_scores_dense[np.arange(len(u_batch)), inverse_indices]
                        except:
                            scores_col[start:end] = model._compute_item_score(u_batch)[np.arange(len(u_batch)), i_batch]
                    else:
                        scores_col[start:end] = model._compute_item_score(u_batch)[np.arange(len(u_batch)), i_batch]

                    pbar.update(1)

            df[f'score_{name}'] = scores_col

        # Calcolo Metriche Relative
        user_grouped = df.groupby('user_id')[f'score_{name}']
        u_mean = user_grouped.transform('mean')
        u_std = user_grouped.transform('std')
        u_max = user_grouped.transform('max')

        df[f'z_score_{name}'] = (df[f'score_{name}'] - u_mean) / (u_std + 1e-9)
        norm_s = df[f'score_{name}'] / (u_max + 1e-9)
        df[f'norm_score_{name}'] = norm_s
        all_norm_scores.append(norm_s.values)

        rank_s = user_grouped.rank(ascending=False, method='min')
        df[f'rank_{name}'] = rank_s
        all_ranks.append(rank_s.values)

        user_count = user_grouped.transform('count')
        df[f'rank_norm_{name}'] = rank_s / user_count


        # ---------------------------------------------------------------------
        # 3. FEATURE SIMILARITY STATS
        # ---------------------------------------------------------------------
        name_lower = name.lower()
        is_target_sim_model = ('rp3' in name_lower) or ('tfidf' in name_lower) or ('itemknn' in name_lower) or ('slim' in name_lower)

        if is_target_sim_model and is_item_item and not is_dense_matrix:
            is_target_skew_kurt = ('rp3' in name_lower) or ('tfidf' in name_lower) or ('slim' in name_lower)
            print(f"      (⚡ Numba SimStats: {name}, skew&kurt: {is_target_skew_kurt})")

            if not isinstance(W_matrix, sps.csc_matrix):
                W_csc = W_matrix.tocsc()
            else:
                W_csc = W_matrix

            W_csc.sort_indices()

            urm_indices = URM_csr.indices
            urm_indptr = URM_csr.indptr

            W_indices = W_csc.indices
            W_indptr = W_csc.indptr
            W_data = W_csc.data

            users_arr = df['user_id'].values.astype(np.int32)
            items_arr = df['item_id'].values.astype(np.int32)

            f_max, f_min, f_mean, f_std, f_skew, f_kurt = compute_sim_stats_numba(
                users_arr, items_arr,
                urm_indices, urm_indptr,
                W_indices, W_indptr, W_data,
                compute_skew_kurt=is_target_skew_kurt
            )

            pfx = f'sim_{name}'
            df[f'{pfx}_max'] = f_max
            df[f'{pfx}_min'] = f_min
            df[f'{pfx}_mean'] = f_mean
            df[f'{pfx}_std'] = f_std

            if is_target_skew_kurt:
                df[f'{pfx}_skew'] = f_skew
                df[f'{pfx}_kurt'] = f_kurt


    # -----------------------------------------------------------------------
    # 4. LATENT FACTORS (PureSVD)
    # -----------------------------------------------------------------------
    if svd_model is not None:
        print(f"   -> Aggiunta feature Latent Factors (primi {n_latent_factors})...", flush=True)
        try:
            u_factors_matrix = getattr(svd_model, 'USER_factors', None)
            i_factors_matrix = getattr(svd_model, 'ITEM_factors', None)

            if u_factors_matrix is None and hasattr(svd_model, 'p_u'): u_factors_matrix = svd_model.p_u
            if i_factors_matrix is None and hasattr(svd_model, 'q_i'): i_factors_matrix = svd_model.q_i

            if u_factors_matrix is not None and i_factors_matrix is not None:
                safe_u_ids_svd = np.clip(user_ids, 0, u_factors_matrix.shape[0]-1)
                safe_i_ids_svd = np.clip(item_ids, 0, i_factors_matrix.shape[0]-1)

                for f in range(n_latent_factors):
                    df[f'user_latent_{f}'] = u_factors_matrix[safe_u_ids_svd, f]
                    df[f'item_latent_{f}'] = i_factors_matrix[safe_i_ids_svd, f]
        except Exception as e:
            print(f"Errore durante estrazione Latent Factors: {e}")


    # -----------------------------------------------------------------------
    # 5. USER STATS
    # -----------------------------------------------------------------------
    try:
        # Nota: URM_csr già calcolata sopra
        item_pop_arr = np.ediff1d(URM_csr.tocsc().indptr).astype(float)
        user_len_arr = np.ediff1d(URM_csr.indptr).astype(float)

        safe_item_ids = np.clip(item_ids, 0, len(item_pop_arr)-1)
        safe_user_ids = np.clip(user_ids, 0, len(user_len_arr)-1)

        df['item_popularity'] = item_pop_arr[safe_item_ids]
        df['user_profile_len'] = user_len_arr[safe_user_ids]

        mask_out_i = item_ids >= len(item_pop_arr); mask_out_u = user_ids >= len(user_len_arr)
        df.loc[mask_out_i, 'item_popularity'] = 0; df.loc[mask_out_u, 'user_profile_len'] = 0

        df['log_item_pop'] = np.log1p(df['item_popularity'])
        df['log_user_len'] = np.log1p(df['user_profile_len'])

        # --- MAINSTREAMNESS & DIVERSITY ---
        print("   -> Calcolo User Mainstreamness & Diversity...", flush=True)
        max_pop = item_pop_arr.max()
        item_pop_norm = item_pop_arr / max_pop if max_pop > 0 else item_pop_arr

        URM_bin = URM_csr.copy()
        URM_bin.data = np.ones_like(URM_bin.data)

        user_pop_sum = URM_bin.dot(item_pop_norm)
        user_pop_sq_sum = URM_bin.dot(item_pop_norm ** 2)

        user_len_safe = user_len_arr.copy()
        user_len_safe[user_len_safe == 0] = 1.0

        user_mainstream = user_pop_sum / user_len_safe
        user_pop_variance = (user_pop_sq_sum / user_len_safe) - (user_mainstream ** 2)
        user_pop_variance[user_pop_variance < 0] = 0
        user_diversity = np.sqrt(user_pop_variance)

        df['user_mainstreamness'] = user_mainstream[safe_user_ids]
        df['user_diversity'] = user_diversity[safe_user_ids]

    except Exception as e:
        print(f"Warning durante calcolo user stats: {e}")
        pass

    print("   -> Calcolo feature Ensemble...", flush=True)
    if all_ranks:
        matrix_ranks = np.array(all_ranks).T
        df['min_rank'] = np.min(matrix_ranks, axis=1)
        df['mean_rank'] = np.mean(matrix_ranks, axis=1)
        df['model_agreement'] = (matrix_ranks <= 20).sum(axis=1)

    if all_norm_scores:
        matrix_norm_scores = np.array(all_norm_scores).T
        df['score_std_dev'] = np.std(matrix_norm_scores, axis=1)

    cols_to_drop = [c for c in df.columns if c.startswith('score_') and c not in ['score_std_dev']]
    df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    return df

In [ ]:
# ==============================================================================
# 3. DATAFRAME GENERATION
# ==============================================================================

# Candidates generator
models_to_generate = {
    'slim': (SLIM_EN_recommender, 140),
    'ials': (iALS_recommender, 150),
    'lightFM': (lightFM_recommender, 140),
    'slim_neg': (SLIM_EN_neg_recommender, 120),
    'rp3sharp': (RP3BetaSharp_recommender, 140),
}

# Feature generators
all_models_dict = {
    'slim': SLIM_EN_recommender,
    'slim_neg': SLIM_EN_neg_recommender,
    'rp3sharp': RP3BetaSharp_recommender,
    'rp3prec': RP3BetaHighPrec_recommender,
    'ials': iALS_recommender,
    'easer': EASE_R_recommender,
    'lightFM': lightFM_recommender,
    'multVAE': multVAE_recommender,
    'TFIDF': TFIDF_recommender,
    'PureSVD': PureSVD_recommender,
}

In [ ]:
# --- TRAINING SET ---
print("\n--- BUILDING TRAINING SET ---")
users_for_xgb = np.unique(URM_xgb.tocoo().row)

# 1. Candidate generation
df_xgb_train = get_candidates_with_labels(
    target_users=users_for_xgb,
    URM_target_labels=URM_xgb,
    candidates_generators_dict=models_to_generate
)

# 2. Feature generation
df_xgb_train = add_features_advanced(df_xgb_train, all_models_dict, URM_train, svd_model=PureSVD_recommender, n_latent_factors=5)

print(f"Dataset XGB creato: {df_xgb_train.shape}")
df_xgb_train.head()

In [ ]:
features_cols = [c for c in df_xgb_train.columns if c not in ['user_id', 'item_id', 'label']]
print(f"\nFeatures usate per il training ({len(features_cols)}):\n{features_cols}")

In [ ]:
df_xgb_train.describe().T

In [ ]:
captured_positives = df_xgb_train[df_xgb_train['label'] == 1].shape[0]
total_ground_truth = URM_xgb.nnz
max_recall = captured_positives / total_ground_truth

print(f"--- ANALISI RECALL CEILING ---")
print(f"Positivi catturati dai candidati: {captured_positives}")
print(f"Totale positivi nel validation set: {total_ground_truth}")
print(f"MAX RECALL TEORICA: {max_recall:.4f}")
print(f"------------------------------")

positives = df_xgb_train['label'].sum()
total = len(df_xgb_train)
print(f"\nBilanciamento Classi:")
print(f"Positivi: {positives} su {total} righe ({positives/total:.2%})")

In [ ]:
import gc

# 1. Type optimization
def optimize_dtypes(df):
    for col in df.columns:
        if 'float' in str(df[col].dtype):
            df[col] = df[col].astype(np.float32)
        elif 'int' in str(df[col].dtype) and df[col].max() < 32000:
             df[col] = df[col].astype(np.int16)
        elif 'int' in str(df[col].dtype):
             df[col] = df[col].astype(np.int32)
    return df

df_xgb_train = optimize_dtypes(df_xgb_train)

# 2. Save on disk
df_xgb_train.to_parquet("df_xgb_train.parquet")

# 3. Remove useless variables
del df_xgb_train
del models_to_generate
del all_models_dict
gc.collect()


# K fold Training

## Stage 2: Reranking with XGBoost & K-Fold Bagging
The second stage uses an **XGBoost Ranker** with a `rank:pairwise` objective to find the optimal order of candidates.

To ensure the model is robust and generalize well, we use **5-Fold Bagging**:
1.  Train 5 separate XGBoost models on different data folds.
2.  Average their predictions during inference.
3.  Monitor the **Pearson Correlation Matrix** between folds to verify that all models have converged to a stable solution.

In [ ]:
xgb_params = {
    'objective': 'rank:pairwise',
    'tree_method': 'hist',
    'device': 'cuda', # If GPU
    'eval_metric': 'map@20',
    'learning_rate': 0.05814195819310938,
     'max_depth': 8,
     'n_estimators': 2500,
     'colsample_bytree': 0.4102815179979831,
     'subsample': 0.9415036029055601,
     'reg_alpha': 6.6012784916341145,
     'reg_lambda': 0.009688337410283411,
     'gamma': 4.126334385757349,
     'min_child_weight': 2
 }

In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
import gc
import os

# K Fold preparation
n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

model_paths = []

import pandas as pd
import numpy as np

DATASET_PATH = "df_xgb_train.parquet"
MODEL_SAVE_DIR = "saved_models_folds"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

df_xgb_train = pd.read_parquet(DATASET_PATH)

df_xgb_train = df_xgb_train.sort_values("user_id")
unique_users_xgb = df_xgb_train['user_id'].unique()

print(f"Inizio Training Bagging su {len(unique_users_xgb)} utenti ({n_folds} folds)...")

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(unique_users_xgb)):
    print(f"\n--- Fold {fold_idx + 1} / {n_folds} ---")

    train_users = unique_users_xgb[train_idx]
    val_users = unique_users_xgb[val_idx]

    mask_train = df_xgb_train['user_id'].isin(train_users)
    mask_val = df_xgb_train['user_id'].isin(val_users)

    X_train = df_xgb_train.loc[mask_train, features_cols]
    y_train = df_xgb_train.loc[mask_train, 'label']
    group_train = df_xgb_train.loc[mask_train].groupby('user_id').size().values

    X_val = df_xgb_train.loc[mask_val, features_cols]
    y_val = df_xgb_train.loc[mask_val, 'label']
    group_val = df_xgb_train.loc[mask_val].groupby('user_id').size().values

    # Train with early stopping
    model = xgb.XGBRanker(**xgb_params)

    model.fit(
        X_train, y_train, group=group_train,
        eval_set=[(X_val, y_val)],
        eval_group=[group_val],
        early_stopping_rounds=50,
        verbose=50
    )

    print(f"   Best Iteration: {model.best_iteration}")

    # Save on disk
    filename = f"{MODEL_SAVE_DIR}/xgb_fold_{fold_idx}.json"
    model.save_model(filename)
    model_paths.append(filename)
    print(f"Modello salvato in: {filename}")

    # Memory cleaning
    del X_train, y_train, group_train
    del X_val, y_val, group_val
    del model
    gc.collect()

In [ ]:
# Data cleaning
del df_xgb_train
gc.collect()

# Base Models Full Training

In [ ]:
from Recommenders.FactorizationMachines.LightFMRecommender import LightFMCFRecommender

lightFM_params = {
                 'loss': 'warp',
                 'n_components': 157,
                 'item_alpha': 0.001019490618869453,
                 'user_alpha': 2.2744836269942957e-05,
                 'learning_rate': 0.012460981535727717,
                 'sgd_mode': 'adadelta',
                 'epochs': 74
              }

lightFM_recommender_full = LightFMCFRecommender(URM_all)
lightFM_recommender_full.fit(**lightFM_params)

In [ ]:
vae_params = {
    'learning_rate': 0.0003696840239724747,
     'l2_reg': 0.0006675069550995708,
     'batch_size': 256,
     'dropout': 0.36764426901664576,
     'total_anneal_steps': 18522,
     'anneal_cap': 0.14211955419238365,
     'encoding_size': 545,
     'next_layer_size_multiplier': 10,
     'max_n_hidden_layers': 1
    }

multVAE_recommender_full = MultVAERecommender_PyTorch_Fixed(URM_all, use_gpu=True)

multVAE_recommender_full.fit(
    epochs=65,
    learning_rate=vae_params['learning_rate'],
    l2_reg=vae_params['l2_reg'],
    batch_size=int(vae_params['batch_size']),
    dropout=vae_params['dropout'],
    anneal_cap=vae_params['anneal_cap'],
    total_anneal_steps=int(vae_params['total_anneal_steps']),
    encoding_size=int(vae_params['encoding_size']),
    next_layer_size_multiplier=int(vae_params['next_layer_size_multiplier']),
    max_n_hidden_layers=int(vae_params['max_n_hidden_layers']),
    sgd_mode='adam'
)

print("Training MultVAE completato con i Best Params.")

In [ ]:
# RP3beta - GRAPH SHARPNER
from Recommenders.Recommender_utils import check_matrix, similarityMatrixTopK
from Recommenders.Similarity.Compute_Similarity_Python import Incremental_Similarity_Builder
from Utils.seconds_to_biggest_unit import seconds_to_biggest_unit
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender

rp3bSharp_params = {"alpha": 1.3, "beta": 0.6, "topK": 200, "normalize_similarity": True}

RP3BetaSharp_recommender_full = RP3betaRecommender(URM_all)
RP3BetaSharp_recommender_full.fit(
    **rp3bSharp_params,
    min_rating = 0,
    implicit = False,
)

In [ ]:
# RP3beta - HIGH PRECISION
from Recommenders.Recommender_utils import check_matrix, similarityMatrixTopK
from Recommenders.Similarity.Compute_Similarity_Python import Incremental_Similarity_Builder
from Utils.seconds_to_biggest_unit import seconds_to_biggest_unit
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender

rp3bHighPrec_params = {"alpha": 0.3, "beta": 0.15, "topK": 40, "normalize_similarity": True}

RP3BetaHighPrec_recommender_full = RP3betaRecommender(URM_all)
RP3BetaHighPrec_recommender_full.fit(
    **rp3bHighPrec_params,
    min_rating = 0,
    implicit = False,
)

In [ ]:
iALS_recommender_full = ImplicitALSRecommender(URM_all, verbose=True)
iALS_recommender_full.fit(
    factors=63,
    regularization=7.535615818925292e-05,
    iterations=82,
    use_gpu=False,
    alpha_val=9.605102175195002,
    dtype=np.float32
)

In [ ]:
#SLIM EN
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender

SLIM_EN_recommender_full = SLIMElasticNetRecommender(URM_all)
SLIM_EN_recommender_full.fit(
    l1_ratio = 0.21208828301360144,
    alpha = 0.0010731928833306275,
    positive_only = True,
    topK = 494
)

In [ ]:
#SLIM EN - NEGATIVES
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender

slimENneg_params = {
            "topK": 200,
            "l1_ratio": 0.1,
            "alpha": 0.001,
            "positive_only": False
        }

SLIM_EN_neg_recommender_full = SLIMElasticNetRecommender(URM_all)
SLIM_EN_neg_recommender_full.fit(
    **slimENneg_params
)



In [ ]:
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender

EASE_R_recommender_full = EASE_R_Recommender(URM_all)
EASE_R_recommender_full.fit(topK=725, l2_norm=375.6601396119732, normalize_matrix=False)


In [ ]:

TFIDF_recommender_full = ImplicitTFIDFRecommender(URM_all, verbose=True)

TFIDF_recommender_full.fit(K=7, num_threads=0)

In [ ]:
#PureSVD
from sklearn.feature_extraction.text import TfidfTransformer
from Recommenders.MatrixFactorization.PureSVDRecommender import PureSVDRecommender
import numpy as np
import scipy.sparse as sps

def start_bm25_transformation(X, k1=1.2, b=0.75):

    X = X.copy()

    N = X.shape[0]

    item_freqs = np.diff(X.tocsc().indptr)
    idf = np.log(N / (item_freqs + 1.0) + 1.0)

    doc_len = np.diff(X.indptr)
    avg_dl = doc_len.mean()

    per_doc_denom = k1 * (1 - b + b * (doc_len / avg_dl))

    per_doc_denom_expanded = np.repeat(per_doc_denom, np.diff(X.indptr))

    tf = X.data

    idf_expanded = idf[X.indices]

    new_data = idf_expanded * ((tf * (k1 + 1)) / (tf + per_doc_denom_expanded))

    X.data = new_data
    return X


k1 = 1.996904472023335
b = 0.12355408697503084
URM_all_transformed = start_bm25_transformation(URM_all, k1=k1, b=b)

PureSVD_recommender_full = PureSVDRecommender(URM_all_transformed, verbose=True)

PureSVD_recommender_full.fit(
    num_factors=121,
    random_seed=1234
)

In [ ]:
import pandas as pd
from tqdm import tqdm

# --- FINAL MODELS DICTIONARY ---

# Candidate generators
models_to_generate = {
    'slim': (SLIM_EN_recommender_full, 140),
    'ials': (iALS_recommender_full, 150),
    'lightFM': (lightFM_recommender_full, 140),
    'slim_neg': (SLIM_EN_neg_recommender_full, 120),
    'rp3sharp': (RP3BetaSharp_recommender_full, 140),
}

# Features generators
all_models_dict = {
    'slim': SLIM_EN_recommender_full,
    'slim_neg': SLIM_EN_neg_recommender_full,
    'rp3sharp': RP3BetaSharp_recommender_full,
    'rp3prec': RP3BetaHighPrec_recommender_full,
    'ials': iALS_recommender_full,
    'easer': EASE_R_recommender_full,
    'lightFM': lightFM_recommender_full,
    'multVAE': multVAE_recommender_full,
    'TFIDF': TFIDF_recommender_full,
    'PureSVD': PureSVD_recommender_full,
}

In [ ]:
# Memory cleaning

import gc
import torch
import ctypes

def clean_memory():
    gc.collect()

    # Free GPU
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

clean_memory()

# Submission

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import glob

# --- CONFIGURATION ---
OUTPUT_NAME = "submission_xgboost_biggerBomb_KFold.csv"
SUBMISSION_CUTOFF = 20
MODEL_SAVE_DIR = "saved_models_folds"

# 1. Load file
target_users_test = pd.read_csv(f"{INPUT_DIR}/data_target_users_test.csv")["user_id"].values

# =============================================================================
# 2. CANDIDATE GENERATION
# =============================================================================
candidates_data = []

print("   -> Generazione candidati dai modelli base...")
for user_id in tqdm(target_users_test):
    candidates = set()

    for name, entry in models_to_generate.items():
        try:
            if isinstance(entry, tuple):
                model, specific_cutoff = entry
            else:                 # Fallback
                model = entry
                specific_cutoff = 50

            recs = model.recommend(user_id, cutoff=specific_cutoff)

            if isinstance(recs, tuple): recs = recs[0]

            candidates.update(recs)
        except Exception as e:
            print(f"Err {name} user {user_id}: {e}")
            continue

    for item_id in candidates:
        candidates_data.append((user_id, item_id))

# Dataframe creation
df_test = pd.DataFrame(candidates_data, columns=['user_id', 'item_id'])
print(f"   -> Candidati generati totali: {len(df_test)} righe.")
print(f"   -> Media candidati per utente: {len(df_test)/len(target_users_test):.1f}")

# =============================================================================
# 3. FEATURE GENERATION
# =============================================================================
df_test = add_features_advanced(df_test, all_models_dict, URM_all, svd_model=PureSVD_recommender_full, n_latent_factors=5)

# NaN check
df_test = df_test.fillna(0)
for col in df_test.columns:
    if 'float' in str(df_test[col].dtype):
        df_test[col] = df_test[col].astype(np.float32)
df_test = df_test.copy()

# =============================================================================
# 4. PREDICTIONS AND K-FOLD
# =============================================================================
print("   -> Predizione Ensemble K-Fold...")

# A. Get models files
model_files = sorted(glob.glob(f"{MODEL_SAVE_DIR}/*.json"))
if not model_files:
    raise FileNotFoundError(f" Nessun modello trovato in {MODEL_SAVE_DIR}!")

print(f"      Trovati {len(model_files)} modelli. Inizio inferenza...")

# B. Initialize to 0
df_test['xgb_score'] = 0.0

# C. For each model
for i, model_path in enumerate(model_files):
    print(f"      [{i+1}/{len(model_files)}] Predizione con {model_path}...", flush=True)

    bst = xgb.Booster()
    bst.load_model(model_path)

    model_features = bst.feature_names

    missing_cols = set(model_features) - set(df_test.columns)
    if missing_cols:
        for c in missing_cols: df_test[c] = 0.0

    dtest = xgb.DMatrix(df_test[model_features])

    preds = bst.predict(dtest)

    df_test['xgb_score'] += preds

    del bst, dtest, preds
    gc.collect()

# D. Folds mean
df_test['xgb_score'] /= len(model_files)
print("Ensemble completato.")

# =============================================================================
# 5. SUBMISSION
# =============================================================================

# TopPop fallback
item_pop = np.ediff1d(URM_all.tocsc().indptr)
top_pop_global = np.argsort(item_pop)[::-1][:200]

submission = []

df_test = df_test.sort_values(['user_id', 'xgb_score'], ascending=[True, False])
df_grouped = df_test.groupby('user_id')

# RECOMMEND
for user_id in tqdm(target_users_test):

    top_items = []

    if user_id in df_grouped.groups:
        group = df_grouped.get_group(user_id)
        top_items = group['item_id'].values[:SUBMISSION_CUTOFF].tolist()

    # Padding (if < 20 recs)
    if len(top_items) < SUBMISSION_CUTOFF:
        start_pos = URM_all.indptr[user_id]
        end_pos = URM_all.indptr[user_id+1]
        seen_items = URM_all.indices[start_pos:end_pos]

        for item in top_pop_global:
            if item not in top_items and item not in seen_items:
                top_items.append(item)
            if len(top_items) >= SUBMISSION_CUTOFF:
                break

    submission.append((user_id, " ".join(map(str, top_items))))

# CSV file
df_submission = pd.DataFrame(submission, columns=['user_id', 'item_list'])
df_submission.to_csv(OUTPUT_NAME, index=False)

## Recovered from errors

It uses smaller batch and further memory cleaning to avoid errors

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import glob
import gc
from tqdm import tqdm

OUTPUT_NAME = "submission_xgboost_biggerBomb_KFold.csv"
MODEL_SAVE_DIR = "saved_models_folds"
CHECKPOINT_PATH = "df_test_checkpoint.parquet"
SUBMISSION_CUTOFF = 20
BATCH_SIZE = 500000

target_users_test = pd.read_csv(f"{INPUT_DIR}/data_target_users_test.csv")["user_id"].values

df_test = pd.read_parquet(CHECKPOINT_PATH)

for col in df_test.columns:
    if df_test[col].dtype == 'float64':
        df_test[col] = df_test[col].astype('float32')

df_test['xgb_score'] = 0.0
gc.collect()

model_files = sorted(glob.glob(f"{MODEL_SAVE_DIR}/*.json"))

for i, model_path in enumerate(model_files):
    print(f"   [{i+1}/{len(model_files)}] Load {model_path}...")

    bst = xgb.Booster()
    bst.load_model(model_path)
    features = bst.feature_names

    missing = set(features) - set(df_test.columns)
    for c in missing: df_test[c] = 0.0

    num_rows = len(df_test)
    for start in tqdm(range(0, num_rows, BATCH_SIZE), desc="      Batch Predict", leave=False):
        end = min(start + BATCH_SIZE, num_rows)

        X_batch = df_test.iloc[start:end][features]

        dtest = xgb.DMatrix(X_batch)

        preds = bst.predict(dtest)

        df_test.iloc[start:end, df_test.columns.get_loc('xgb_score')] += preds

        del X_batch, dtest, preds

    del bst
    gc.collect()

df_test['xgb_score'] /= len(model_files)


item_pop = np.ediff1d(URM_all.tocsc().indptr)
top_pop_global = np.argsort(item_pop)[::-1][:200]

submission = []
df_test.sort_values(['user_id', 'xgb_score'], ascending=[True, False], inplace=True)
df_grouped = df_test.groupby('user_id')

for user_id in tqdm(target_users_test):
    top_items = []

    if user_id in df_grouped.groups:
        top_items = df_test.loc[df_grouped.groups[user_id], 'item_id'].values[:SUBMISSION_CUTOFF].tolist()

    if len(top_items) < SUBMISSION_CUTOFF:
        seen = URM_all.indices[URM_all.indptr[user_id]:URM_all.indptr[user_id+1]]
        for item in top_pop_global:
            if item not in top_items and item not in seen:
                top_items.append(item)
            if len(top_items) >= SUBMISSION_CUTOFF: break

    submission.append((user_id, " ".join(map(str, top_items))))

pd.DataFrame(submission, columns=['user_id', 'item_list']).to_csv(OUTPUT_NAME, index=False)

# Folds Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import glob
import json
import os

MODEL_SAVE_DIR = "saved_models_folds"

model_files = sorted(glob.glob(f"{MODEL_SAVE_DIR}/*.json"))

if not model_files:
    raise ValueError(f"Errore: Nessun modello JSON trovato in '{MODEL_SAVE_DIR}'.")

fold_stats = []
feature_importance_list = []


for i, model_path in enumerate(model_files):
    bst = xgb.Booster()
    bst.load_model(model_path)

    attrs = bst.attributes()

    if 'best_iteration' in attrs:
        best_trees = int(attrs['best_iteration'])
    else:
        best_trees = bst.num_boosted_rounds()

    if 'best_score' in attrs:
        best_score = float(attrs['best_score'])
    else:
        best_score = np.nan

    fi_dict = bst.get_score(importance_type='gain')
    feature_importance_list.append(fi_dict)

    fold_stats.append({
        'Fold': i + 1,
        'MAP@20': best_score,
        'Best_Trees': best_trees,
        'Filename': os.path.basename(model_path)
    })

df_stats = pd.DataFrame(fold_stats)

print("\n--- 1. Dettaglio per Fold ---")
if df_stats['MAP@20'].isna().all():
    print(df_stats[['Fold', 'Best_Trees', 'Filename']].to_string(index=False))
else:
    print(df_stats.to_string(index=False))

mean_trees = df_stats['Best_Trees'].mean()

print("\n--- 2. Statistiche Globali (Ensemble) ---")
if not df_stats['MAP@20'].isna().all():
    mean_score = df_stats['MAP@20'].mean()
    std_score = df_stats['MAP@20'].std()
    cv_score = (std_score / mean_score) * 100 if mean_score > 0 else 0
    print(f"Average MAP@20:    {mean_score:.5f}")
    print(f"Std Dev (Risk):    {std_score:.5f}")
    print(f"Stability (CV%):   {cv_score:.2f}%")
else:
    print("Average MAP@20:    N/A (Dati non presenti nel JSON)")

print(f"Avg Trees Used:    {mean_trees:.1f}")

plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
if not df_stats['MAP@20'].isna().all():
    bars = plt.bar(df_stats['Fold'], df_stats['MAP@20'], color='skyblue', edgecolor='black')
    plt.axhline(mean_score, color='red', linestyle='--', label=f'Mean: {mean_score:.4f}')

    y_min = df_stats['MAP@20'].min()
    y_max = df_stats['MAP@20'].max()
    if pd.notna(y_min) and pd.notna(y_max) and y_min > 0:
        plt.ylim(y_min * 0.98, y_max * 1.02)

    plt.title('MAP@20 Score per Fold')
    plt.xlabel('Fold')
    plt.ylabel('MAP@20')
    plt.legend()
    plt.grid(axis='y', alpha=0.3)

    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval, f"{yval:.4f}", va='bottom', ha='center', fontsize=9)
else:
    plt.text(0.5, 0.5, "Score Data Not Available in JSON", ha='center', va='center')
    plt.title('MAP@20 Score (Non disponibile)')

plt.subplot(1, 2, 2)
plt.plot(df_stats['Fold'], df_stats['Best_Trees'], marker='o', linestyle='-', color='green')
plt.title('Numero di Alberi (Early Stopping point)')
plt.xlabel('Fold')
plt.ylabel('N Estimators')
plt.grid(True, alpha=0.3)

for i, txt in enumerate(df_stats['Best_Trees']):
    plt.annotate(txt, (df_stats['Fold'][i], df_stats['Best_Trees'][i]), textcoords="offset points", xytext=(0,10), ha='center')

plt.tight_layout()
plt.show()

print("\n--- 3. Top 5 Feature per Fold (Consistenza) ---")

for i, fi_dict in enumerate(feature_importance_list):
    if not fi_dict:
        print(f"Fold {i+1}: Nessuna feature importance trovata.")
        continue

    sorted_fi = sorted(fi_dict.items(), key=lambda x: x[1], reverse=True)

    top_5_names = [k for k, v in sorted_fi[:5]]

    print(f"Fold {i+1}: {top_5_names}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import xgboost as xgb
import glob
import gc

MODEL_SAVE_DIR = "saved_models_folds"

model_files = sorted(glob.glob(f"{MODEL_SAVE_DIR}/*.json"))
if not model_files:
    raise ValueError(f"Nessun modello trovato in {MODEL_SAVE_DIR}!")

sample_size = 10000
if len(df_test) > sample_size:
    df_sample = df_test.sample(n=sample_size, random_state=42).copy()
else:
    df_sample = df_test.copy()

if 'features_cols' not in locals():
    temp_bst = xgb.Booster()
    temp_bst.load_model(model_files[0])
    features_cols = temp_bst.feature_names
    del temp_bst

X_sample = df_sample[features_cols]

pred_matrix = {}

for i, model_path in enumerate(model_files):
    bst = xgb.Booster()
    bst.load_model(model_path)

    dmatrix_sample = xgb.DMatrix(X_sample)
    preds = bst.predict(dmatrix_sample)

    pred_matrix[f'Fold_{i+1}'] = preds

    del bst, dmatrix_sample, preds
    gc.collect()

df_preds = pd.DataFrame(pred_matrix)

corr_matrix = df_preds.corr(method='pearson')

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=0.8, vmax=1.0, fmt=".3f")
plt.title('Matrice di Correlazione tra i Fold (Pearson)')
plt.show()

avg_corr = corr_matrix.mean().mean()
print(f"Correlazione Media tra i fold: {avg_corr:.4f}")

df_sample['pred_std'] = df_preds.std(axis=1)
df_sample['pred_mean'] = df_preds.mean(axis=1)

top_disagreement = df_sample.sort_values('pred_std', ascending=False).head(5)

print("\n--- Esempi di Massimo Disaccordo (Dove i modelli litigano) ---")
cols_to_show = ['user_id', 'item_id', 'pred_std', 'pred_mean']
print(top_disagreement[cols_to_show])

plt.figure(figsize=(10, 4))
sns.histplot(df_sample['pred_std'], bins=50, kde=True, color='orange')
plt.title('Distribuzione del Disaccordo (Std Dev delle predizioni)')
plt.xlabel('Deviazione Standard dei Score')
plt.show()

if len(model_files) >= 2:
    test_user = df_sample['user_id'].iloc[0]
    user_data = df_test[df_test['user_id'] == test_user].copy()

    if len(user_data) > 0:
        print(f"\n--- Confronto Top-10 per l'utente {test_user} (Fold 1 vs Fold 2) ---")
        X_user = user_data[features_cols]
        dmatrix_user = xgb.DMatrix(X_user)

        bst1 = xgb.Booster()
        bst1.load_model(model_files[0])
        score_m1 = bst1.predict(dmatrix_user)
        del bst1

        bst2 = xgb.Booster()
        bst2.load_model(model_files[1])
        score_m2 = bst2.predict(dmatrix_user)
        del bst2
        gc.collect()

        user_data['score_m1'] = score_m1
        user_data['score_m2'] = score_m2

        top_m1 = set(user_data.sort_values('score_m1', ascending=False).head(10)['item_id'].values)
        top_m2 = set(user_data.sort_values('score_m2', ascending=False).head(10)['item_id'].values)

        intersection = top_m1.intersection(top_m2)
        print(f"Item in comune: {len(intersection)} su 10")
        print(f"Jaccard Similarity: {len(intersection) / len(top_m1.union(top_m2)):.2f}")
else:
    print("\nImpossibile fare confronto Top-N: serve più di 1 modello salvato.")

#Final Model Analysis
Our final implementation demonstrates the effectiveness of a high-performance two-stage pipeline, achieving a Public Recall@20 of 0.53331 and a Private Recall@20 of 0.53149.

##Feature Insights and Engineering
The Feature Importance (Gain) analysis confirms that while base model scores—specifically the z_score_slim are dominant primary signals, they are not sufficient on their own. The integration of Model Agreement and advanced Similarity Statistics provided the necessary refinement to scale the leaderboard. By using the Numba library to compute statistical moments like skewness and max similarity across millions of user-item pairs, we provided the XGBoost reranker with deep insights into the distribution of similarity weights within each user's history.

##Robustness and K-Fold Stability
The extremely high correlation between folds (0.998) observed in our 5-fold cross-validation demonstrates that our solution is exceptionally stable and not reliant on a specific data split. This high level of consistency across all folds proves that the model has learned generalized patterns of user behavior, ensuring robust performance and minimizing the risk of overfitting to specific training samples.